# Coding practice: interpolate and regress a flow

Flow matching connects two distributions by pairing a sample from each, moving in a straight line from
one to the other, and training a network to predict the velocity that carries the first endpoint to the
second. No noise is added along the way. Build the interpolants and targets below, then compare a pairing that starts from noise with one that starts from related data.

> Save your own copy first: File → Save a copy in Drive.

<details style="border:1px solid #e5e7eb;border-radius:8px;padding:10px 14px;background:#f9fafb;color:#111827;margin:14px 0;">
<summary style="cursor:pointer;font-weight:600;">Hint: the formulas</summary>

$$x_t=(1-t)\,x_0+t\,x_1,\qquad \text{target}=x_1-x_0.$$

</details>

In [ ]:
import math
import random

random.seed(0)


def interpolate(x0, x1, t):
    """Return the flow-matching input x_t for endpoints x0, x1 at time t in [0, 1]."""
    if not 0.0 <= t <= 1.0:
        raise ValueError("t is a fraction of the way from x0 to x1, so it lies in [0, 1].")
    # TODO: return x_t for this pair at time t.
    raise NotImplementedError("Return x_t.")


def regression_target(x0, x1):
    """Return the quantity the network is trained to predict for this pair."""
    # TODO: return the regression target for this pair.
    raise NotImplementedError("Return the regression target.")

In [ ]:
x0, x1 = 2.0, 10.0
for t in (0.0, 0.25, 0.5, 0.75, 1.0):
    print(f"t={t:>4}: x_t={interpolate(x0, x1, t):>6.2f}   target={regression_target(x0, x1):>6.2f}")

# x_t should start at x0, end at x1, and sit halfway between them at t = 0.5.
assert abs(interpolate(x0, x1, 0.0) - x0) < 1e-12
assert abs(interpolate(x0, x1, 1.0) - x1) < 1e-12
assert abs(interpolate(x0, x1, 0.5) - 0.5 * (x0 + x1)) < 1e-12
# Following the target from x_t for the rest of the time should land exactly on x1.
assert abs(interpolate(x0, x1, 0.25) + 0.75 * regression_target(x0, x1) - x1) < 1e-12
print("\nChecks passed.")

In [ ]:
# Two pairings with the same recipe: one starts from noise, the other from a
# blurred copy of each sharp sample.
def make_pairs(source, n=6):
    sharp = [random.gauss(8.0, 1.0) for _ in range(n)]          # the target distribution
    if source == "noise":
        start = [random.gauss(0.0, 1.0) for _ in range(n)]
    elif source == "blurry":
        start = [value + random.gauss(0.0, 0.3) for value in sharp]  # a coupled, related sample
    else:
        raise ValueError("source must be 'noise' or 'blurry'")
    return list(zip(start, sharp))


for source in ("noise", "blurry"):
    pairs = make_pairs(source)
    spread = [abs(regression_target(a, b)) for a, b in pairs]
    print(f"{source:>7} start: mean |target| = {sum(spread) / len(spread):.2f}")

## Interpret the results

Answer in your notes:

1. In the table for one pair, what changes as `t` moves and what stays the same? What kind of training problem does that make this?
2. Compare the mean target size for the two pairings. What does it say about how much the model has to learn in each case?
3. Which single line of `make_pairs` separates the two cases?
4. Nothing here adds noise to `x_t`. Where does the variety the model trains on come from?